<a href="https://colab.research.google.com/github/sdilsaath24/communication_analysis/blob/main/DILSAATH_2303717610622084_ECE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# DIGITAL COMMUNICATION CHANNEL SIMULATOR
# GOOGLE COLAB - CLICKABLE BUTTON VERSION
# ============================================================

# Install required libraries
!pip install -q numpy matplotlib pandas ipywidgets


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# GLOBAL VARIABLES
# ============================================================

scenarios = []

simulation_results = None

application_running = True


# ============================================================
# 1. BPSK MODULATION
# ============================================================

def bpsk_modulate(bits):

    # 0 -> -1
    # 1 -> +1

    return 2 * bits - 1


def bpsk_demodulate(received):

    bits = (received.real >= 0).astype(int)

    return bits


# ============================================================
# 2. QPSK MODULATION
# ============================================================

def qpsk_modulate(bits):

    # Make number of bits even

    original_length = len(bits)

    if len(bits) % 2 != 0:

        bits = np.append(bits, 0)

    symbols = []

    for i in range(0, len(bits), 2):

        b1 = bits[i]
        b2 = bits[i + 1]

        # Gray coding

        if b1 == 0 and b2 == 0:

            symbol = 1 + 1j

        elif b1 == 0 and b2 == 1:

            symbol = -1 + 1j

        elif b1 == 1 and b2 == 1:

            symbol = -1 - 1j

        else:

            symbol = 1 - 1j

        symbols.append(symbol)

    symbols = np.array(symbols)

    # Normalize average power

    symbols = symbols / np.sqrt(2)

    return symbols


def qpsk_demodulate(received):

    bits = []

    for symbol in received:

        real = symbol.real
        imag = symbol.imag

        if real >= 0 and imag >= 0:

            bits.extend([0, 0])

        elif real < 0 and imag >= 0:

            bits.extend([0, 1])

        elif real < 0 and imag < 0:

            bits.extend([1, 1])

        else:

            bits.extend([1, 0])

    return np.array(bits)


# ============================================================
# 3. 16-QAM MODULATION
# ============================================================

def qam16_modulate(bits):

    # Make number of bits multiple of 4

    remainder = len(bits) % 4

    if remainder != 0:

        bits = np.append(
            bits,
            np.zeros(4 - remainder, dtype=int)
        )

    groups = bits.reshape(-1, 4)

    symbols = []

    # Gray-coded mapping

    mapping = {

        (0, 0): -3,

        (0, 1): -1,

        (1, 1): 1,

        (1, 0): 3

    }

    for group in groups:

        b1, b2, b3, b4 = group

        i_level = mapping[(b1, b2)]

        q_level = mapping[(b3, b4)]

        symbol = i_level + 1j * q_level

        symbols.append(symbol)

    symbols = np.array(symbols)

    # Normalize average power

    symbols = symbols / np.sqrt(10)

    return symbols


def qam16_demodulate(received):

    # Undo normalization

    received = received * np.sqrt(10)

    bits = []

    def level_to_bits(level):

        if level < -2:

            return [0, 0]

        elif level < 0:

            return [0, 1]

        elif level < 2:

            return [1, 1]

        else:

            return [1, 0]

    for symbol in received:

        real_bits = level_to_bits(
            symbol.real
        )

        imag_bits = level_to_bits(
            symbol.imag
        )

        bits.extend(real_bits)

        bits.extend(imag_bits)

    return np.array(bits)


# ============================================================
# 4. BITS PER SYMBOL
# ============================================================

def get_bits_per_symbol(modulation):

    if modulation == "BPSK":

        return 1

    elif modulation == "QPSK":

        return 2

    elif modulation == "16-QAM":

        return 4

    return 1


# ============================================================
# 5. MODULATION FUNCTION
# ============================================================

def modulate(bits, modulation):

    if modulation == "BPSK":

        return bpsk_modulate(bits)

    elif modulation == "QPSK":

        return qpsk_modulate(bits)

    elif modulation == "16-QAM":

        return qam16_modulate(bits)

    else:

        raise ValueError(
            "Invalid modulation"
        )


# ============================================================
# 6. DEMODULATION FUNCTION
# ============================================================

def demodulate(received, modulation):

    if modulation == "BPSK":

        return bpsk_demodulate(received)

    elif modulation == "QPSK":

        return qpsk_demodulate(received)

    elif modulation == "16-QAM":

        return qam16_demodulate(received)

    else:

        raise ValueError(
            "Invalid modulation"
        )


# ============================================================
# 7. AWGN CHANNEL
# ============================================================

def add_awgn_noise(signal, snr_db):

    # Convert SNR from dB to linear

    snr_linear = 10 ** (snr_db / 10)

    # Calculate signal power

    signal_power = np.mean(
        np.abs(signal) ** 2
    )

    # Calculate noise power

    noise_power = (
        signal_power / snr_linear
    )

    # Standard deviation of complex noise

    noise_std = np.sqrt(
        noise_power / 2
    )

    # Generate complex Gaussian noise

    noise = (

        np.random.normal(
            0,
            noise_std,
            len(signal)
        )

        +

        1j * np.random.normal(
            0,
            noise_std,
            len(signal)
        )

    )

    # Received signal

    received = signal + noise

    return received


# ============================================================
# 8. BER SIMULATION
# ============================================================

def simulate_ber(
    modulation,
    snr_db,
    number_of_bits
):

    # Generate random bits

    bits = np.random.randint(
        0,
        2,
        number_of_bits
    )

    # Modulation

    transmitted = modulate(
        bits,
        modulation
    )

    # Add AWGN noise

    received = add_awgn_noise(
        transmitted,
        snr_db
    )

    # Demodulation

    received_bits = demodulate(
        received,
        modulation
    )

    # Compare only the available bits

    usable_bits = min(
        len(bits),
        len(received_bits)
    )

    original_bits = bits[
        :usable_bits
    ]

    recovered_bits = received_bits[
        :usable_bits
    ]

    # Count errors

    errors = np.sum(
        original_bits != recovered_bits
    )

    # BER

    ber = errors / usable_bits

    return ber


# ============================================================
# 9. SNR CALCULATION
# ============================================================

def calculate_snr(
    signal_power,
    noise_power
):

    if noise_power <= 0:

        return float("inf")

    snr = 10 * math.log10(
        signal_power / noise_power
    )

    return snr


# ============================================================
# 10. PACKET LOSS
# ============================================================

def calculate_packet_loss(
    ber,
    packet_size
):

    if ber >= 1:

        return 100.0

    probability_success = (
        (1 - ber) ** packet_size
    )

    packet_loss = (
        1 - probability_success
    )

    return packet_loss * 100


# ============================================================
# 11. RUN ONE SCENARIO
# ============================================================

def run_scenario(scenario):

    signal_power = scenario[
        "Signal Power"
    ]

    noise_power = scenario[
        "Noise Power"
    ]

    bandwidth = scenario[
        "Bandwidth"
    ]

    packet_size = scenario[
        "Packet Size"
    ]

    modulation = scenario[
        "Modulation"
    ]

    number_of_bits = scenario[
        "Number of Bits"
    ]

    # SNR

    snr_db = calculate_snr(
        signal_power,
        noise_power
    )

    # BER

    ber = simulate_ber(
        modulation,
        snr_db,
        number_of_bits
    )

    # Packet loss

    packet_loss = calculate_packet_loss(
        ber,
        packet_size
    )

    # Bits per symbol

    bits_per_symbol = get_bits_per_symbol(
        modulation
    )

    # Simplified symbol rate ≈ bandwidth

    raw_throughput = (
        bandwidth *
        bits_per_symbol
    )

    # Effective throughput

    effective_throughput = (
        raw_throughput *
        (1 - ber)
    )

    return {

        "Signal Power (W)": signal_power,

        "Noise Power (W)": noise_power,

        "Bandwidth (Hz)": bandwidth,

        "Modulation": modulation,

        "SNR (dB)": snr_db,

        "BER": ber,

        "Packet Loss (%)": packet_loss,

        "Raw Throughput (bps)": raw_throughput,

        "Effective Throughput (bps)":
            effective_throughput

    }


# ============================================================
# 12. RUN ALL SCENARIOS
# ============================================================

def run_all_scenarios():

    global simulation_results

    if len(scenarios) == 0:

        return None

    results = []

    for index, scenario in enumerate(
        scenarios
    ):

        result = run_scenario(
            scenario
        )

        result[
            "Scenario"
        ] = index + 1

        results.append(result)

    simulation_results = pd.DataFrame(
        results
    )

    # Arrange columns

    simulation_results = simulation_results[

        [

            "Scenario",

            "Modulation",

            "Signal Power (W)",

            "Noise Power (W)",

            "Bandwidth (Hz)",

            "SNR (dB)",

            "BER",

            "Packet Loss (%)",

            "Raw Throughput (bps)",

            "Effective Throughput (bps)"

        ]

    ]

    return simulation_results


# ============================================================
# 13. DISPLAY COMPARISON
# ============================================================

def display_comparison():

    if simulation_results is None:

        print(
            "Please run the scenarios first."
        )

        return

    print("=" * 90)

    print(
        "SIMULATION COMPARISON"
    )

    print("=" * 90)

    display(
        simulation_results.style.format({

            "SNR (dB)": "{:.2f}",

            "BER": "{:.6f}",

            "Packet Loss (%)": "{:.2f}",

            "Raw Throughput (bps)": "{:.2f}",

            "Effective Throughput (bps)":
                "{:.2f}"

        })
    )

    print("\n")

    print("=" * 70)

    print(
        "DIGITAL COMMUNICATION TRADE-OFF"
    )

    print("=" * 70)

    print("""
BPSK
• 1 bit per symbol
• Lower throughput
• Strong robustness against noise

QPSK
• 2 bits per symbol
• Higher throughput than BPSK
• Moderate complexity

16-QAM
• 4 bits per symbol
• Higher potential throughput
• More sensitive to noise

GENERAL TRADE-OFF

Higher-order modulation can transmit more
bits per symbol and therefore provide higher
potential throughput.

However, higher-order modulation generally
requires better channel conditions to achieve
a low BER.

Therefore:

Higher modulation order
        ↓
Higher data rate
        ↓
Greater sensitivity to noise
""")


# ============================================================
# 14. BER VS SNR GRAPH
# ============================================================

def plot_ber_vs_snr(
    number_of_bits
):

    snr_values = np.arange(
        0,
        21,
        2
    )

    modulations = [

        "BPSK",

        "QPSK",

        "16-QAM"

    ]

    plt.figure(
        figsize=(9, 6)
    )

    for modulation in modulations:

        ber_values = []

        print(
            f"Simulating {modulation}..."
        )

        for snr in snr_values:

            ber = simulate_ber(

                modulation,

                snr,

                number_of_bits

            )

            # Prevent log(0)

            ber = max(
                ber,
                1e-6
            )

            ber_values.append(
                ber
            )

        plt.semilogy(

            snr_values,

            ber_values,

            marker="o",

            label=modulation

        )

    plt.xlabel(
        "SNR (dB)"
    )

    plt.ylabel(
        "Bit Error Rate (BER)"
    )

    plt.title(
        "BER vs SNR"
    )

    plt.grid(True)

    plt.legend()

    plt.show()

    plt.close()


# ============================================================
# 15. THROUGHPUT VS SNR GRAPH
# ============================================================

def plot_throughput_vs_snr(
    bandwidth,
    number_of_bits
):

    snr_values = np.arange(
        0,
        21,
        2
    )

    modulations = [

        "BPSK",

        "QPSK",

        "16-QAM"

    ]

    plt.figure(
        figsize=(9, 6)
    )

    for modulation in modulations:

        bits_per_symbol = (
            get_bits_per_symbol(
                modulation
            )
        )

        raw_throughput = (
            bandwidth *
            bits_per_symbol
        )

        throughput_values = []

        print(
            f"Simulating {modulation}..."
        )

        for snr in snr_values:

            ber = simulate_ber(

                modulation,

                snr,

                number_of_bits

            )

            effective_throughput = (

                raw_throughput *

                (1 - ber)

            )

            throughput_values.append(
                effective_throughput
            )

        plt.plot(

            snr_values,

            throughput_values,

            marker="o",

            label=modulation

        )

    plt.xlabel(
        "SNR (dB)"
    )

    plt.ylabel(
        "Effective Throughput (bps)"
    )

    plt.title(
        "Throughput vs SNR"
    )

    plt.grid(True)

    plt.legend()

    plt.show()

    plt.close()


# ============================================================
# 16. GUI OUTPUT AREA
# ============================================================

output = widgets.Output()


# ============================================================
# 17. SCENARIO INPUT WIDGETS
# ============================================================

signal_power_widget = widgets.FloatText(

    value=10,

    description="Signal Power:",

    style={
        "description_width": "130px"
    }

)

noise_power_widget = widgets.FloatText(

    value=1,

    description="Noise Power:",

    style={
        "description_width": "130px"
    }

)

bandwidth_widget = widgets.FloatText(

    value=1000,

    description="Bandwidth (Hz):",

    style={
        "description_width": "130px"
    }

)

packet_size_widget = widgets.IntText(

    value=100,

    description="Packet Size:",

    style={
        "description_width": "130px"
    }

)

modulation_widget = widgets.Dropdown(

    options=[
        "BPSK",
        "QPSK",
        "16-QAM"
    ],

    value="BPSK",

    description="Modulation:",

    style={
        "description_width": "130px"
    }

)

simulation_bits_widget = widgets.IntText(

    value=10000,

    description="Simulation Bits:",

    style={
        "description_width": "130px"
    }

)


# ============================================================
# 18. SCENARIO BUTTONS
# ============================================================

add_scenario_button = widgets.Button(

    description="Add This Scenario",

    button_style="success",

    icon="plus",

    layout=widgets.Layout(
        width="180px"
    )

)

cancel_scenario_button = widgets.Button(

    description="Back to Menu",

    button_style="warning",

    layout=widgets.Layout(
        width="150px"
    )

)


# ============================================================
# 19. GRAPH WIDGETS
# ============================================================

ber_bits_widget = widgets.IntText(

    value=10000,

    description="Simulation Bits:",

    style={
        "description_width": "130px"
    }

)

ber_generate_button = widgets.Button(

    description="Generate BER Graph",

    button_style="success"

)

ber_back_button = widgets.Button(

    description="Back to Menu",

    button_style="warning"

)


throughput_bandwidth_widget = widgets.FloatText(

    value=1000,

    description="Bandwidth (Hz):",

    style={
        "description_width": "130px"
    }

)

throughput_bits_widget = widgets.IntText(

    value=10000,

    description="Simulation Bits:",

    style={
        "description_width": "130px"
    }

)

throughput_generate_button = widgets.Button(

    description="Generate Throughput Graph",

    button_style="success"

)

throughput_back_button = widgets.Button(

    description="Back to Menu",

    button_style="warning"

)


# ============================================================
# 20. MAIN MENU BUTTONS
# ============================================================

button1 = widgets.Button(

    description="1. Add Scenario",

    layout=widgets.Layout(
        width="200px",
        height="45px"
    )

)

button2 = widgets.Button(

    description="2. Run Scenarios",

    layout=widgets.Layout(
        width="200px",
        height="45px"
    )

)

button3 = widgets.Button(

    description="3. Comparison",

    layout=widgets.Layout(
        width="200px",
        height="45px"
    )

)

button4 = widgets.Button(

    description="4. BER vs SNR",

    layout=widgets.Layout(
        width="200px",
        height="45px"
    )

)

button5 = widgets.Button(

    description="5. Throughput vs SNR",

    layout=widgets.Layout(
        width="200px",
        height="45px"
    )

)

button6 = widgets.Button(

    description="6. Exit",

    button_style="danger",

    layout=widgets.Layout(
        width="200px",
        height="45px"
    )

)


# ============================================================
# 21. MAIN MENU DISPLAY
# ============================================================

def show_main_menu():

    clear_output(wait=True)

    title = widgets.HTML(

        value="""
        <div style="
            text-align:center;
            padding:15px;
            border-radius:10px;
            background:#eeeeee;
        ">

        <h1>
        Digital Communication Channel Simulator
        </h1>

        <p>
        Compare BPSK, QPSK and 16-QAM
        </p>

        </div>
        """

    )

    scenario_count = widgets.HTML(

        value=f"""
        <h3>
        Scenarios Added: {len(scenarios)}
        </h3>
        """

    )

    menu = widgets.VBox([

        title,

        scenario_count,

        widgets.HTML(
            "<h3>Choose an operation:</h3>"
        ),

        widgets.HBox([

            button1,

            button2,

            button3

        ]),

        widgets.HBox([

            button4,

            button5,

            button6

        ]),

        output

    ])

    display(menu)


# ============================================================
# 22. BUTTON 1 - ADD SCENARIO
# ============================================================

def button1_clicked(b):

    clear_output(wait=True)

    form_title = widgets.HTML(

        value="""
        <h2>Add Simulation Scenario</h2>

        <p>
        Enter the channel parameters below.
        </p>
        """

    )

    form = widgets.VBox([

        form_title,

        signal_power_widget,

        noise_power_widget,

        bandwidth_widget,

        packet_size_widget,

        modulation_widget,

        simulation_bits_widget,

        widgets.HBox([

            add_scenario_button,

            cancel_scenario_button

        ]),

        output

    ])

    display(form)


# ============================================================
# 23. ADD SCENARIO
# ============================================================

def add_scenario_clicked(b):

    with output:

        clear_output(wait=True)

        signal_power = (
            signal_power_widget.value
        )

        noise_power = (
            noise_power_widget.value
        )

        bandwidth = (
            bandwidth_widget.value
        )

        packet_size = (
            packet_size_widget.value
        )

        modulation = (
            modulation_widget.value
        )

        number_of_bits = (
            simulation_bits_widget.value
        )

        # Validation

        if signal_power <= 0:

            print(
                "❌ Signal power must be greater than 0."
            )

            return

        if noise_power <= 0:

            print(
                "❌ Noise power must be greater than 0."
            )

            return

        if bandwidth <= 0:

            print(
                "❌ Bandwidth must be greater than 0."
            )

            return

        if packet_size <= 0:

            print(
                "❌ Packet size must be greater than 0."
            )

            return

        if number_of_bits <= 0:

            print(
                "❌ Simulation bits must be greater than 0."
            )

            return

        # Create scenario

        scenario = {

            "Signal Power":
                signal_power,

            "Noise Power":
                noise_power,

            "Bandwidth":
                bandwidth,

            "Packet Size":
                packet_size,

            "Modulation":
                modulation,

            "Number of Bits":
                number_of_bits

        }

        scenarios.append(
            scenario
        )

        print("=" * 60)

        print(
            "SCENARIO ADDED SUCCESSFULLY"
        )

        print("=" * 60)

        print(
            f"\nScenario Number: "
            f"{len(scenarios)}"
        )

        print(
            f"Signal Power: "
            f"{signal_power} W"
        )

        print(
            f"Noise Power: "
            f"{noise_power} W"
        )

        print(
            f"Bandwidth: "
            f"{bandwidth} Hz"
        )

        print(
            f"Packet Size: "
            f"{packet_size} bits"
        )

        print(
            f"Modulation: "
            f"{modulation}"
        )

        print(
            f"Simulation Bits: "
            f"{number_of_bits}"
        )

        print(
            "\n✅ Scenario saved."
        )

        print(
            "\nClick 'Back to Menu' to continue."
        )


# ============================================================
# 24. BUTTON 2 - RUN SCENARIOS
# ============================================================

def button2_clicked(b):

    clear_output(wait=True)

    title = widgets.HTML(
        value="<h2>Running Simulations...</h2>"
    )

    display(title)

    if len(scenarios) == 0:

        print(
            "❌ No scenarios have been added."
        )

        print(
            "\nPlease click "
            "'1. Add Scenario'."
        )

        return

    print(
        f"Running {len(scenarios)} scenario(s)..."
    )

    print()

    for index, scenario in enumerate(
        scenarios
    ):

        print(

            f"Scenario {index + 1}: "
            f"{scenario['Modulation']}"

        )

    print(
        "\nPlease wait..."
    )

    run_all_scenarios()

    print(
        "\n✅ All simulations completed."
    )

    print(
        "\nClick '3. Comparison' "
        "to see the results."
    )

    back_button = widgets.Button(

        description="Back to Menu",

        button_style="warning"

    )

    display(back_button)

    def back_clicked(b):

        show_main_menu()

    back_button.on_click(
        back_clicked
    )


# ============================================================
# 25. BUTTON 3 - COMPARISON
# ============================================================

def button3_clicked(b):

    clear_output(wait=True)

    print("=" * 80)

    print(
        "SIMULATION COMPARISON"
    )

    print("=" * 80)

    if simulation_results is None:

        print(
            "\n❌ No results available."
        )

        print(
            "\nPlease click "
            "'2. Run Scenarios' first."
        )

    else:

        display_comparison()

    print()

    back_button = widgets.Button(

        description="Back to Menu",

        button_style="warning"

    )

    display(back_button)

    back_button.on_click(
        lambda b: show_main_menu()
    )


# ============================================================
# 26. BUTTON 4 - BER FORM
# ============================================================

def button4_clicked(b):

    clear_output(wait=True)

    title = widgets.HTML(

        value="""
        <h2>BER vs SNR</h2>

        <p>
        Enter the number of random bits
        used for the BER simulation.
        </p>
        """

    )

    form = widgets.VBox([

        title,

        ber_bits_widget,

        widgets.HBox([

            ber_generate_button,

            ber_back_button

        ]),

        output

    ])

    display(form)


# ============================================================
# 27. GENERATE BER GRAPH
# ============================================================

def ber_generate_clicked(b):

    with output:

        clear_output(wait=True)

        number_of_bits = (
            ber_bits_widget.value
        )

        if number_of_bits <= 0:

            print(
                "❌ Number of bits must be greater than 0."
            )

            return

        print(
            "Generating BER vs SNR graph..."
        )

        print(
            "This may take a few seconds..."
        )

        print()

        plot_ber_vs_snr(
            number_of_bits
        )

        print(
            "\n✅ BER graph generated."
        )

        print(
            "\nClick 'Back to Menu'."
        )


# ============================================================
# 28. BUTTON 5 - THROUGHPUT FORM
# ============================================================

def button5_clicked(b):

    clear_output(wait=True)

    title = widgets.HTML(

        value="""
        <h2>Throughput vs SNR</h2>

        <p>
        Enter the bandwidth and number
        of simulation bits.
        </p>
        """

    )

    form = widgets.VBox([

        title,

        throughput_bandwidth_widget,

        throughput_bits_widget,

        widgets.HBox([

            throughput_generate_button,

            throughput_back_button

        ]),

        output

    ])

    display(form)


# ============================================================
# 29. GENERATE THROUGHPUT GRAPH
# ============================================================

def throughput_generate_clicked(b):

    with output:

        clear_output(wait=True)

        bandwidth = (
            throughput_bandwidth_widget.value
        )

        number_of_bits = (
            throughput_bits_widget.value
        )

        if bandwidth <= 0:

            print(
                "❌ Bandwidth must be greater than 0."
            )

            return

        if number_of_bits <= 0:

            print(
                "❌ Number of bits must be greater than 0."
            )

            return

        print(
            "Generating Throughput vs SNR graph..."
        )

        print(
            "This may take a few seconds..."
        )

        print()

        plot_throughput_vs_snr(

            bandwidth,

            number_of_bits

        )

        print(
            "\n✅ Throughput graph generated."
        )

        print(
            "\nClick 'Back to Menu'."
        )


# ============================================================
# 30. BUTTON 6 - EXIT
# ============================================================

def button6_clicked(b):

    clear_output(wait=True)

    print("=" * 60)

    print(
        "DIGITAL COMMUNICATION SIMULATOR"
    )

    print("=" * 60)

    print(
        "\nSimulator closed."
    )

    print(
        "\nThank you!"
    )


# ============================================================
# 31. CONNECT BUTTONS
# ============================================================

button1.on_click(
    button1_clicked
)

button2.on_click(
    button2_clicked
)

button3.on_click(
    button3_clicked
)

button4.on_click(
    button4_clicked
)

button5.on_click(
    button5_clicked
)

button6.on_click(
    button6_clicked
)

add_scenario_button.on_click(
    add_scenario_clicked
)

cancel_scenario_button.on_click(
    lambda b: show_main_menu()
)

ber_generate_button.on_click(
    ber_generate_clicked
)

ber_back_button.on_click(
    lambda b: show_main_menu()
)

throughput_generate_button.on_click(
    throughput_generate_clicked
)

throughput_back_button.on_click(
    lambda b: show_main_menu()
)


# ============================================================
# 32. START APPLICATION
# ============================================================

show_main_menu()